# 09 — Four-way soft MoE with reduced dataset-specific warm-start

This alternative full-data experiment keeps the complete three-stage MoE method while reducing the expensive initialization schedule. Stage A learns a shared representation for 3 epochs, Stage B **mandatorily warm-starts each of the four experts only on its own dataset** for 2 epochs, and Stage C jointly fine-tunes the soft-gated MoE for at most 30 epochs with validation macro-F1 early stopping. During Stage C, every row still uses all four experts in the soft mixture, but only its dataset-owned expert receives an expert-parameter gradient. The shared encoder and task-driven soft gate continue learning across all datasets.

Run the notebook top to bottom in a GPU Colab runtime. Add a Colab Secret named `GITHUB_TOKEN` with read access to the private repository and grant this notebook access to it. Prepared split artifacts and compatible checkpoints are safely reused after interruptions.


In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = "selimsidan"
GITHUB_REPO = "dataset_moe_nids"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"

DRIVE_DATA_DIR = "/content/drive/MyDrive/NIDS_datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs"
EXECUTION_MODE = "out_of_core_full"  # out_of_core_full | in_memory_smoke

ACTIVE_DATASETS = [
    "NF-UNSW-NB15-v3",
    "NF-ToN-IoT-v3",
    "NF-BoT-IoT-v3",
    "NF-CICIDS2018-v3",
]
ARCHITECTURE = "moe_dataset_soft"
RUN_NAME = "nfv3_4way_moe_soft_owned_experts_seed0_v1"
SEED = 0

# Comparable compact capacity: shared 47→128→64 encoder, four linear
# 64→classes experts, and a soft 64→4 gate.
LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
EXPERT_HIDDEN_DIMS = []
DROPOUT = 0.2
GATE_SUPERVISION = "none"
LAMBDA_BALANCE = 0.1
EXPERT_UPDATE_POLICY = "assigned_only"  # mandatory in this notebook
LAMBDA_EXPERT_ANCHOR = 0.0001  # small pull toward Stage-B expert parameters

# Reduced warm-start schedule. Stage B must remain greater than zero:
# each expert is trained exclusively on its corresponding dataset.
EPOCHS_A = 3
EPOCHS_B = 2
EPOCHS_C = 30
BATCH_SIZE = 512
PROGRESS_EVERY_ROWS = 1_000_000
FORCE_RESTART = False
RUN_TESTS = True
# ==================================================================


## 1. Secure checkout and environment setup


In [ ]:
import base64, os, subprocess, sys, time
from pathlib import Path
try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook is intended for Google Colab.") from exc
drive.mount("/content/drive")
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f"Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ | {
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
}
repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
repo_dir = Path("/content") / GITHUB_REPO
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(repo_dir)], env=git_env, check=True)
git_env.clear()
token = auth = None
os.chdir(repo_dir)
os.environ["NIDS_DRIVE_BASE"] = DRIVE_DATA_DIR
os.environ["NIDS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
os.environ["NIDS_SCRATCH_DIR"] = "/content/dataset_moe_nids_scratch"
os.environ["PYTHONUNBUFFERED"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Repository:", repo_dir)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Preflight and tests

This verifies the GPU, four files, shared 47-feature schema, and mandatory Stage B setting before expensive preparation begins.


In [ ]:
import shutil
import torch
from data.registry import get_spec
if EXECUTION_MODE not in {"out_of_core_full", "in_memory_smoke"}:
    raise ValueError("Unknown EXECUTION_MODE")
if ARCHITECTURE != "moe_dataset_soft":
    raise ValueError("Notebook 09 is specifically for moe_dataset_soft")
if len(ACTIVE_DATASETS) != 4 or len(set(ACTIVE_DATASETS)) != 4:
    raise ValueError("Notebook 09 requires four distinct datasets")
if EPOCHS_B <= 0:
    raise ValueError("Stage B is mandatory: EPOCHS_B must be greater than zero")
if EXPERT_UPDATE_POLICY != "assigned_only":
    raise ValueError("Notebook 09 requires assigned_only expert updates in Stage C")
if LAMBDA_EXPERT_ANCHOR < 0:
    raise ValueError("LAMBDA_EXPERT_ANCHOR must be non-negative")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU.")
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    if not any(Path(path).is_file() for path in spec.paths):
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError("Missing selected datasets:\n" + "\n".join(f"  {k}: {v}" for k, v in missing.items()))
aliases = [get_spec(name).feature_alias for name in ACTIVE_DATASETS]
if any(get_spec(name).kind != "file" for name in ACTIVE_DATASETS) or any(value != aliases[0] for value in aliases[1:]):
    raise ValueError("The full-data combination must use schema-compatible single-file NF-v3 datasets")
if len(aliases[0]) != 47:
    raise ValueError(f"Expected 47 features, found {len(aliases[0])}")
local_disk = shutil.disk_usage("/content")
print("GPU:", torch.cuda.get_device_name(0))
print("Datasets:", ACTIVE_DATASETS)
print(f"Schedule: Stage A={EPOCHS_A}, Stage B={EPOCHS_B} per expert, Stage C≤{EPOCHS_C}")
print("Stage B specialization:", dict(zip(range(1, 5), ACTIVE_DATASETS)))
print("Stage C expert ownership:", EXPERT_UPDATE_POLICY, "| Stage-B anchor:", LAMBDA_EXPERT_ANCHOR)
print(f"Local disk: {local_disk.free / 2**30:.1f} GiB free of {local_disk.total / 2**30:.1f} GiB")
print("Run:", RUN_NAME, "| resume enabled:", not FORCE_RESTART)


In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-u", "-m", "pytest", "-q"], check=True, env=os.environ.copy())
else:
    print("Tests skipped by configuration.")


## 3. Prepare, train Stage A → B → C, and evaluate

The process is deliberately unbuffered. It reports preparation activity immediately and training progress every `PROGRESS_EVERY_ROWS` rows. Stage B trains each expert exclusively on its assigned dataset. In Stage C, the forward prediction remains the normal four-expert soft mixture, but the backward path into non-owned experts is detached per row. The gate receives the unchanged final classification gradient, the shared encoder remains trainable on the pooled data, and a small anchor discourages drift from the Stage-B expert parameters. Stage checkpoints are saved after every completed epoch; keep `FORCE_RESTART=False` to resume after a Colab interruption.


In [ ]:
overrides = [
    f"run_name={RUN_NAME}",
    f"seed={SEED}",
    f"architecture={ARCHITECTURE}",
    "data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]",
    "training.device=cuda",
    f"training.force_restart={str(FORCE_RESTART).lower()}",
    f"training.epochs_a={EPOCHS_A}",
    f"training.epochs_b={EPOCHS_B}",
    f"training.epochs_c={EPOCHS_C}",
    f"training.batch_size={BATCH_SIZE}",
    f"training.progress_every_rows={PROGRESS_EVERY_ROWS}",
    f"model.latent_dim={LATENT_DIM}",
    "model.encoder.hidden_dims=[" + ",".join(map(str, ENCODER_HIDDEN_DIMS)) + "]",
    "model.expert.hidden_dims=[" + ",".join(map(str, EXPERT_HIDDEN_DIMS)) + "]",
    f"model.encoder.dropout={DROPOUT}",
    f"model.expert.dropout={DROPOUT}",
    f"training.stage_c.gate_supervision={GATE_SUPERVISION}",
    f"training.stage_c.expert_update_policy={EXPERT_UPDATE_POLICY}",
    f"training.stage_c.lambda_expert_anchor={LAMBDA_EXPERT_ANCHOR}",
    "training.stage_c_unfreeze=all",
    f"load_balance.lambda_balance={LAMBDA_BALANCE}",
]
module = "training.ooc_run" if EXECUTION_MODE == "out_of_core_full" else "training.run"
cmd = [sys.executable, "-u", "-m", module, "--config", "config/default.yaml"]
if EXECUTION_MODE == "in_memory_smoke":
    cmd += ["--mode", "smoke"]
for override in overrides:
    cmd += ["--set", override]
run_started = time.monotonic()
print("Launching unbuffered:", module, flush=True)
subprocess.run(cmd, check=True, env=os.environ.copy())
print(f"Training and evaluation completed in {(time.monotonic() - run_started) / 3600:.2f} hours.")


## 4. Review persisted detailed results


In [ ]:
import pandas as pd
from IPython.display import display
result_dir = Path(DRIVE_OUTPUT_DIR) / "results" / RUN_NAME
overall = pd.read_csv(result_dir / "Overall_Metrics.csv")
per_class = pd.read_csv(result_dir / "Per_Class_Metrics.csv")
print("=== Overall and per-origin metrics ===")
display(overall)
print("=== Native per-class metrics, weakest F1 first ===")
native = per_class[per_class["is_native_class"]] if "is_native_class" in per_class.columns else per_class
display(native.sort_values(["origin", "f1", "support"]).reset_index(drop=True))
for filename in ["Gate_By_Dataset.csv", "Expert_Performance_By_Dataset.csv", "Expert_Utilization.csv", "Confusion_Matrix.csv"]:
    path = result_dir / filename
    if path.is_file():
        print(f"=== {filename} ===")
        display(pd.read_csv(path))
print("Detailed artifacts:", result_dir)
print("Checkpoints:", Path(DRIVE_OUTPUT_DIR) / "checkpoints" / RUN_NAME)


## Interpretation and recovery

- This is a complete four-expert soft MoE: Stage B explicitly assigns and trains one expert per dataset, then Stage C combines all four through the learned soft gate.
- Expert ownership persists in Stage C: expert `d` receives classification gradients only from rows originating in dataset `d`. Non-owned expert outputs remain visible to the gate, so the forward prediction and gate learning remain genuinely soft.
- Dataset identity is never an inference input. `GATE_SUPERVISION=none` also means it does not directly supervise the gate during training; it is used only to enforce expert-gradient ownership.
- `LAMBDA_EXPERT_ANCHOR` controls the optional mean-squared pull toward the Stage-B expert bank. Set it to `0.0` to disable anchoring while retaining assigned-only updates.
- `Expert_Performance_By_Dataset.csv` evaluates every expert independently on every dataset and marks its assigned dataset. Training ownership is guaranteed by the gradient mask; diagonal performance dominance is an empirical outcome reported here, not assumed.
- The schedule uses about 35 pooled-data-equivalent training passes instead of the original 70: 3 pooled Stage-A passes, 2 aggregate Stage-B passes, and at most 30 Stage-C passes.
- Stage C may stop earlier after five validation epochs without macro-F1 improvement. Stages A and B intentionally run their configured fixed epoch counts.
- If Colab disconnects, reconnect and run the notebook again with `FORCE_RESTART=False`. Prepared splits and completed epoch checkpoints will be reused.
- Change both `SEED` and `RUN_NAME` for later independent seeds. Do not reuse a run name with changed training settings.
